# 🏥 Pipeline ETL — Banco de Preços em Saúde (BPS)

## Descrição
Este notebook implementa o pipeline **ETL (Extract, Transform, Load)** para integração
e consolidação das bases do Banco de Preços em Saúde (BPS) referentes aos anos de
**2023, 2024 e 2025**.

O resultado final é um **Data Warehouse modelado em Esquema Estrela** carregado em um
banco de dados PostgreSQL local, contendo as seguintes tabelas:
- `dim_produto` — informações sobre os itens comprados
- `dim_instituicao` — hospitais e entidades compradoras
- `dim_fornecedor` — empresas fornecedoras
- `dim_tempo` — datas e períodos das compras
- `fato_compras` — registros de compras com métricas e chaves estrangeiras

## Fonte dos dados
- **Base:** Banco de Preços em Saúde (BPS)
- **Anos utilizados:** 2023, 2024 e 2025

## 1. Instalação de dependências e imports

Para executar este pipeline, precisamos das seguintes bibliotecas:

| Biblioteca | Finalidade |
|------------|------------|
| `pandas` | Leitura, manipulação e transformação dos dados em DataFrames |
| `sqlalchemy` | Interface de conexão com o banco de dados PostgreSQL |
| `psycopg2-binary` | Driver do PostgreSQL para Python (usado internamente pelo SQLAlchemy) |
| `python-dotenv` | Leitura das credenciais do banco a partir do arquivo `.env` |

> ⚠️ **Atenção:** certifique-se de que o arquivo `.env` está criado na raiz do projeto
> antes de executar as células de conexão. Veja o modelo em `.env.example`.

In [1]:
# SEÇÃO 1 — Instalação de dependências e imports

# Instalação das bibliotecas necessárias
# O flag -q suprime o output verbose do pip, deixando o notebook mais limpo
%pip install pandas sqlalchemy psycopg2-binary python-dotenv -q

# ── Imports ──────────────────────────────────────────────────
import pandas as pd                        # Manipulação de dados
from sqlalchemy import create_engine, text # Conexão com PostgreSQL
from dotenv import load_dotenv             # Leitura do .env
import os                                  # Acesso às variáveis de ambiente

print("✅ Bibliotecas instaladas e importadas com sucesso!")
print(f"   Pandas versão: {pd.__version__}")

Note: you may need to restart the kernel to use updated packages.
✅ Bibliotecas instaladas e importadas com sucesso!
   Pandas versão: 3.0.3


## 2. Extract — Extração dos dados

A etapa de **Extração** consiste em ler as bases brutas do BPS referentes aos anos de
2023, 2024 e 2025 a partir dos arquivos CSV disponibilizados pelo Ministério da Saúde.

Os arquivos devem estar na pasta `data/` na raiz do repositório, com os seguintes nomes:
- `data/2023.csv`
- `data/2024.csv`
- `data/2025.csv`

Após a leitura, realizamos uma **exploração inicial** de cada base para entender:
- Quantidade de linhas e colunas
- Tipos de dados de cada coluna
- Presença de valores nulos
- Amostra dos primeiros registros

In [2]:
# SEÇÃO 2 — EXTRACT

# Dicionário com os caminhos dos arquivos
# A chave é o ano — usada para criar a coluna ano_origem
arquivos = {
    '2023': '../data/raw/2023.csv',
    '2024': '../data/raw/2024.csv',
    '2025': '../data/raw/2025.csv'
}

lista_dataframes = []

for ano, arquivo in arquivos.items():
    try:
        df_temp = pd.read_csv(arquivo, sep=';', encoding='latin-1')
        df_temp['ano_origem'] = ano  # Rastreabilidade do ano de origem
        lista_dataframes.append(df_temp)
        print(f"✅ {ano}: {df_temp.shape[0]:,} linhas | {df_temp.shape[1]} colunas")
    except Exception as e:
        print(f"❌ Erro ao ler {ano}: {e}")

# Concatenação de todos os anos em um único DataFrame
df = pd.concat(lista_dataframes, ignore_index=True)
print(f"\n📊 Total unificado: {df.shape[0]:,} linhas | {df.shape[1]} colunas")

✅ 2023: 31,992 linhas | 26 colunas
✅ 2024: 26,258 linhas | 26 colunas
✅ 2025: 26,215 linhas | 26 colunas

📊 Total unificado: 84,465 linhas | 26 colunas


### Exploração inicial da base unificada

Com os dados de todos os anos em um único DataFrame, fazemos uma
inspeção geral antes de qualquer transformação.

In [3]:
# Visão geral
print("=== INFORMAÇÕES GERAIS ===")
df.info()

print("\n=== PRIMEIRAS LINHAS ===")
display(df.head(5))

print("\n=== VALORES ÚNICOS POR COLUNA ===")
for col in df.columns:
    print(f"{col}: {df[col].nunique()} únicos")

print("\n=== NULOS POR COLUNA ===")
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumo_nulos = pd.DataFrame({'nulos': nulos, '%': nulos_pct})
print(resumo_nulos[resumo_nulos['nulos'] > 0])

=== INFORMAÇÕES GERAIS ===
<class 'pandas.DataFrame'>
RangeIndex: 84465 entries, 0 to 84464
Data columns (total 26 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   ano_compra                       84465 non-null  int64  
 1   nome_instituicao                 84316 non-null  str    
 2   esfera                           84465 non-null  str    
 3   cnpj_instituicao                 84465 non-null  str    
 4   municipio_instituicao            84465 non-null  str    
 5   uf                               84465 non-null  str    
 6   compra                           84465 non-null  str    
 7   insercao                         82457 non-null  str    
 8   codigo_br                        84465 non-null  int64  
 9   descricao_catmat                 84465 non-null  str    
 10  unidade_fornecimento             84454 non-null  str    
 11  generico                         54492 non-null  str    
 12  an

,ano_compra,nome_instituicao,esfera,cnpj_instituicao,municipio_instituicao,uf,compra,insercao,codigo_br,descricao_catmat,...,unidade_medida,unidade_fornecimento_capacidade,cnpj_fornecedor,fornecedor,cnpj_fabricante,fabricante,qtd_itens_comprados,preco_unitario,preco_total,ano_origem
0,2023,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18.478.187/0001-07,MARABA,PA,01/01/2023,15/03/2024,423975,"PIPETA, TIPO:PASTEUR, CAPACIDADE:3 ML, MATERIA...",...,NaN,UNIDADE,05.323.167/0001-07,CIRUBEL COMERCIO E REPRESENTACOES DE PRODUTOS ...,57.893.521/0001-32,PERFITECNICA PERFIS TECNICOS DE BORRACHA LTDA,36,45.00,1620.0,2023
1,2023,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18.478.187/0001-07,MARABA,PA,01/01/2023,15/03/2024,457503,"CORANTE, TIPO :PARDO DE BISMARCK, CARACTERÃST...",...,NaN,UNIDADE,07.944.100/0001-15,PROC9 INDUSTRIA QUIMICA LTDA,07.944.100/0001-15,PROC9 INDUSTRIA QUIMICA LTDA,4,190.00,760.0,2023
2,2023,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18.478.187/0001-07,MARABA,PA,01/01/2023,15/03/2024,382447,"REAGENTE PARA DIAGNÃSTICO CLÃNICO 5, TIPO:AL...",...,ML,FRASCO 10.00 ML,05.048.534/0001-01,NORTEMED DISTRIBUIDORA DE PRODUTOS MEDICOS LTDA,50.657.402/0001-31,EBRAM PRODUTOS LABORATORIAIS LTDA,20,33.38,667.6,2023
3,2023,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18.478.187/0001-07,MARABA,PA,01/01/2023,15/03/2024,470305,"MEIO DE CULTURA - CÃLULA E TECIDO, TIPO:MEIO ...",...,NaN,GRAMA,05.323.167/0001-07,CIRUBEL COMERCIO E REPRESENTACOES DE PRODUTOS ...,73.636.391/0001-09,NEWPROV PRODUTOS PARA LABORATORIO LTDA,80,25.00,2000.0,2023
4,2023,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18.478.187/0001-07,MARABA,PA,01/01/2023,15/03/2024,479641,"ADAPTADOR USO MÃDICO, APLICAÃÃO:P/ AGULHA D...",...,NaN,UNIDADE,05.048.534/0001-01,NORTEMED DISTRIBUIDORA DE PRODUTOS MEDICOS LTDA,48.740.849/0001-28,CRAL ARTIGOS PARA LABORATORIO LTDA,50,38.71,1935.5,2023



=== VALORES ÚNICOS POR COLUNA ===
ano_compra: 3 únicos
nome_instituicao: 318 únicos
esfera: 2 únicos
cnpj_instituicao: 407 únicos
municipio_instituicao: 363 únicos
uf: 21 únicos
compra: 892 únicos
insercao: 693 únicos
codigo_br: 7354 únicos
descricao_catmat: 7353 únicos
unidade_fornecimento: 41 únicos
generico: 2 únicos
anvisa: 9000 únicos
modalidade_compra: 8 únicos
tipo_compra: 2 únicos
capacidade: 115 únicos
unidade_medida: 12 únicos
unidade_fornecimento_capacidade: 324 únicos
cnpj_fornecedor: 1929 únicos
fornecedor: 1788 únicos
cnpj_fabricante: 1262 únicos
fabricante: 1163 únicos
qtd_itens_comprados: 10325 únicos
preco_unitario: 14256 únicos
preco_total: 34717 únicos
ano_origem: 3 únicos

=== NULOS POR COLUNA ===
                                 nulos      %
nome_instituicao                   149   0.18
insercao                          2008   2.38
unidade_fornecimento                11   0.01
generico                         29973  35.49
anvisa                           29973  35

### Análise da base unificada

#### Visão geral
- **Total de registros:** 84.465 linhas × 26 colunas (25 originais + `ano_origem`)
- **Período coberto:** 2023, 2024 e 2025 (3 únicos em `ano_compra`)
- **Instituições:** 407 CNPJs únicos, distribuídos em 21 UFs e 363 municípios
- **Produtos:** 7.354 códigos CATMAT únicos
- **Fornecedores:** 1.929 CNPJs únicos de fornecedores

#### Colunas com valores nulos — decisões de tratamento

| Coluna | Nulos | % | Decisão |
|---|---|---|---|
| `generico` | ~29.973 | 35,49% | Preencher com `'Não informado'` |
| `anvisa` | ~29.973 | 35,49% | Preencher com `0` (sem registro) |
| `capacidade` | 53.068 | 62,83% | Preencher com `0.0` (não aplicável) |
| `unidade_medida` | 53.068 | 62,83% | Preencher com `'Não informado'` |
| `insercao` | 2.008 | 2,38% | Preencher com `'Não informado'` |
| `nome_instituicao` | 149 | 0,18% | Preencher com `'Não informado'` |
| `unidade_fornecimento` | 11 | 0,01% | Preencher com `'Não informado'` |
| `unidade_fornecimento_capacidade` | 11 | 0,01% | Preencher com `'Não informado'` |

> **Observação:** `capacidade` e `unidade_medida` possuem 62,83% de nulos pois
> esses campos só se aplicam a itens com capacidade definida (ex: frascos de 500ml).
> Para os demais itens, a ausência é esperada e não representa erro nos dados.

> **Observação:** `generico` e `anvisa` com ~35% de nulos indicam que parte dos
> itens são dispositivos médicos ou insumos, que não possuem classificação de
> genérico nem registro Anvisa obrigatório.

#### Colunas que precisam de conversão de tipo

| Coluna | Tipo atual | Tipo esperado | Motivo |
|---|---|---|---|
| `compra` | `str` | `datetime` | Data em formato BR (`dd/mm/yyyy`) |
| `insercao` | `str` | `datetime` | Data em formato BR (`dd/mm/yyyy`) |
| `cnpj_instituicao` | `str` | `str` limpa | Pode conter pontuação (`XX.XXX.XXX/XXXX-XX`) |
| `cnpj_fornecedor` | `str` | `str` limpa | Mesma situação |
| `cnpj_fabricante` | `str` | `str` limpa | Mesma situação |
| `preco_unitario` | `str` | `float` | Vírgula decimal (`"1.234,56"`) |
| `preco_total` | `str` | `float` | Mesma situação |

#### Observações sobre cardinalidade
- `fornecedor` tem 1.788 únicos vs `cnpj_fornecedor` com 1.929 — indica que
  alguns CNPJs diferentes compartilham o mesmo nome (filiais ou variações de grafia).
  A chave natural da `dim_fornecedor` será o **CNPJ**, não o nome.
- `descricao_catmat` tem 7.353 únicos vs `codigo_br` com 7.354 — um código sem
  descrição. Será investigado na etapa de limpeza.

### Inspeção de valores únicos nas colunas de interesse

Antes de transformar, inspecionamos os valores únicos das colunas categóricas
mais relevantes para entender o domínio dos dados e identificar possíveis
inconsistências de grafia ou padronização.

In [4]:
colunas_interesse = [
    'esfera',
    'generico',
    'modalidade_compra',
    'tipo_compra',
    'unidade_medida',
    'unidade_fornecimento'
]

for col in colunas_interesse:
    print(f"\n=== {col.upper()} ===")
    print(df[col].value_counts(dropna=False).to_string())


=== ESFERA ===
esfera
MUNICIPAL    74325
ESTADUAL     10140

=== GENERICO ===
generico
NaN    29973
S      28789
N      25703

=== MODALIDADE_COMPRA ===
modalidade_compra
PregÃ£o                           71565
Registro de PreÃ§os                9469
Dispensa de LicitaÃ§Ã£o            3158
ConcorrÃªncia                        88
Tomada de PreÃ§os                    75
Inexigibilidade de LicitaÃ§Ã£o       67
LeilÃ£o                              33
Concurso                             10

=== TIPO_COMPRA ===
tipo_compra
ADMINISTRATIVA    79386
JUDICIAL           5079

=== UNIDADE_MEDIDA ===
unidade_medida
NaN      53068
ML       25461
G         3342
UN        1059
DOSES      871
M          491
L           92
KG          40
MG          19
MCL         18
KUI          2
CM           1
DOSE         1

=== UNIDADE_FORNECIMENTO ===
unidade_fornecimento
COMPRIMIDO               31888
FRASCO                   14185
UNIDADE                  10414
AMPOLA                   10135
CÃPSULA          

### Problemas identificados na inspeção

#### 1. Encoding corrompido em `modalidade_compra` e `unidade_fornecimento`
Caracteres especiais do português aparecem corrompidos devido a conflito de encoding
na leitura dos CSVs. Exemplos:
- `"PregÃ£o"` → deveria ser `"Pregão"`
- `"ConcorrÃªncia"` → deveria ser `"Concorrência"`
- `"CÃPSULA"` → deveria ser `"CÁPSULA"`

**Correção:** reaplicar encoding correto (`latin-1` → `utf-8`) ou usar `.encode('latin-1').decode('utf-8')` nas colunas afetadas.

#### 2. Duplicidade em `unidade_medida`
- `"DOSES"` e `"DOSE"` representam a mesma unidade — serão unificadas para `"DOSES"`.

#### 3. Duplicidade em `unidade_fornecimento`
- `"DOSE"` aparece separado das demais — será unificado para `"UNIDADE"` ou mantido conforme análise.

#### 4. Nulos em `generico`
- 29.973 nulos (35,49%) — representam itens sem classificação de genérico
  (dispositivos médicos, insumos). Serão preenchidos com `"Não informado"`.

#### 5. `esfera` sem problemas
- Apenas 2 categorias (`MUNICIPAL`, `ESTADUAL`), sem inconsistências.

#### 6. `tipo_compra` sem problemas
- Apenas 2 categorias (`ADMINISTRATIVA`, `JUDICIAL`), sem inconsistências.

In [5]:
# Verificar se a correção de encoding funciona
teste = df['modalidade_compra'].dropna().unique()
print("Antes:")
print(teste)

print("\nDepois:")
corrigido = [v.encode('latin-1').decode('utf-8') for v in teste]
print(corrigido)

Antes:
<StringArray>
[                       'PregÃ£o',        'Dispensa de LicitaÃ§Ã£o',
            'Registro de PreÃ§os',              'Tomada de PreÃ§os',
 'Inexigibilidade de LicitaÃ§Ã£o',                  'ConcorrÃªncia',
                        'LeilÃ£o',                       'Concurso']
Length: 8, dtype: str

Depois:
['Pregão', 'Dispensa de Licitação', 'Registro de Preços', 'Tomada de Preços', 'Inexigibilidade de Licitação', 'Concorrência', 'Leilão', 'Concurso']


In [6]:
# Identificar todas as colunas str com encoding corrompido
# O padrão do problema é a presença de 'Ã' no texto
colunas_str = df.select_dtypes(include='str').columns

print("Colunas com possível encoding corrompido:")
for col in colunas_str:
    valores = df[col].dropna().astype(str)
    tem_problema = valores.str.contains('Ã', na=False).any()
    if tem_problema:
        exemplo = valores[valores.str.contains('Ã', na=False)].iloc[0]
        print(f"  {col}: ex → '{exemplo}'")

Colunas com possível encoding corrompido:
  descricao_catmat: ex → 'PIPETA, TIPO:PASTEUR, CAPACIDADE:3 ML, MATERIAL:PLÃSTICO, TIPO USO:DESCARTÃVEL'
  unidade_fornecimento: ex → 'CÃPSULA'
  modalidade_compra: ex → 'PregÃ£o'
  unidade_fornecimento_capacidade: ex → 'CÃPSULA'


## 3. Transform — Transformação dos dados

A etapa de **Transformação** é responsável por preparar os dados brutos para
a modelagem dimensional. Ela é dividida em três sub-etapas:

1. **Limpeza e padronização** — tratamento de nulos, conversão de tipos,
   padronização de CNPJs e datas
2. **Unificação e deduplicação** — garantir que o DataFrame unificado está
   consistente entre os anos
3. **Modelagem dimensional** — separar o DataFrame em tabelas dimensão
   (`dim_produto`, `dim_instituicao`, `dim_fornecedor`, `dim_tempo`)
   e tabela fato (`fato_compras`), criando as chaves surrogate que
   conectam tudo no Esquema Estrela

### 3.1 Limpeza e padronização

Aplicamos as seguintes correções no DataFrame unificado:

- **Encoding corrompido:** 4 colunas com caracteres especiais mal interpretados
  (`descricao_catmat`, `unidade_fornecimento`, `modalidade_compra`,
  `unidade_fornecimento_capacidade`) — corrigidas com `.encode('latin-1').decode('utf-8')`
- **Nulos textuais:** preenchidos com `'Não informado'`
- **Nulos numéricos:** preenchidos com `0`
- **Datas:** convertidas de string BR (`dd/mm/yyyy`) para `datetime`
- **Preços:** convertidos de string com vírgula decimal para `float`
- **CNPJs:** removida pontuação, mantidos apenas dígitos
- **Duplicidade:** `'DOSE'` unificado para `'DOSES'` em `unidade_medida`

In [7]:
# SEÇÃO 3 — TRANSFORM

# ── 3.1 Limpeza e padronização ───────────────────────────────

df_clean = df.copy()

# -- Encoding: corrigir colunas com latin-1 corrompido ----------
colunas_encoding = [
    'descricao_catmat',
    'unidade_fornecimento',
    'modalidade_compra',
    'unidade_fornecimento_capacidade'
]

for col in colunas_encoding:
    df_clean[col] = df_clean[col].apply(
        lambda x: x.encode('latin-1').decode('utf-8') if isinstance(x, str) else x
    )

# -- Nulos: colunas de texto ------------------------------------
colunas_texto_nulo = [
    'nome_instituicao',
    'unidade_fornecimento',
    'generico',
    'unidade_medida',
    'unidade_fornecimento_capacidade'
]
df_clean[colunas_texto_nulo] = df_clean[colunas_texto_nulo].fillna('Não informado')

# -- Nulos: colunas numéricas -----------------------------------
df_clean['anvisa']    = df_clean['anvisa'].fillna(0)
df_clean['capacidade'] = df_clean['capacidade'].fillna(0.0)

# -- Datas: converter string BR para datetime -------------------
df_clean['compra']   = pd.to_datetime(df_clean['compra'],   format='%d/%m/%Y', errors='coerce')
df_clean['insercao'] = pd.to_datetime(df_clean['insercao'], format='%d/%m/%Y', errors='coerce')

# -- Preços: vírgula decimal → float ----------------------------
def limpar_valor(valor):
    if isinstance(valor, str):
        return float(valor.replace('.', '').replace(',', '.'))
    return valor

df_clean['preco_unitario']      = df_clean['preco_unitario'].apply(limpar_valor)
df_clean['preco_total']         = df_clean['preco_total'].apply(limpar_valor)
df_clean['qtd_itens_comprados'] = pd.to_numeric(
    df_clean['qtd_itens_comprados'], errors='coerce'
).fillna(0)

# -- CNPJs: remover pontuação -----------------------------------
def limpar_cnpj(cnpj):
    if isinstance(cnpj, str):
        return ''.join(filter(str.isdigit, cnpj))
    return cnpj

df_clean['cnpj_instituicao'] = df_clean['cnpj_instituicao'].apply(limpar_cnpj)
df_clean['cnpj_fornecedor']  = df_clean['cnpj_fornecedor'].apply(limpar_cnpj)
df_clean['cnpj_fabricante']  = df_clean['cnpj_fabricante'].apply(limpar_cnpj)

# -- Duplicidade: unificar DOSE → DOSES em unidade_medida ------
df_clean['unidade_medida'] = df_clean['unidade_medida'].replace('DOSE', 'DOSES')

# -- Verificação final ------------------------------------------
print("✅ Limpeza concluída!")

nulos = df_clean.isnull().sum()
nulos_restantes = nulos[nulos > 0]

if len(nulos_restantes) == 0:
    print("   Nulos restantes: 0")
else:
    print(f"   Nulos restantes por coluna:")
    print(nulos_restantes)

print(f"\nTipos das colunas convertidas:")
print(df_clean[['compra', 'insercao', 'preco_unitario', 'preco_total', 'qtd_itens_comprados']].dtypes)
print(f"\nModalidade após correção de encoding:")
print(df_clean['modalidade_compra'].unique())

✅ Limpeza concluída!
   Nulos restantes por coluna:
insercao    2008
dtype: int64

Tipos das colunas convertidas:
compra                 datetime64[us]
insercao               datetime64[us]
preco_unitario                float64
preco_total                   float64
qtd_itens_comprados             int64
dtype: object

Modalidade após correção de encoding:
<StringArray>
[                      'Pregão',        'Dispensa de Licitação',
           'Registro de Preços',             'Tomada de Preços',
 'Inexigibilidade de Licitação',                 'Concorrência',
                       'Leilão',                     'Concurso']
Length: 8, dtype: str


In [8]:
# -- Remover colunas redundantes --------------------------------
df_clean = df_clean.drop(columns=['ano_compra', 'ano_origem'])

print(f"Colunas restantes: {df_clean.shape[1]}")
print(df_clean.columns.tolist())

Colunas restantes: 24
['nome_instituicao', 'esfera', 'cnpj_instituicao', 'municipio_instituicao', 'uf', 'compra', 'insercao', 'codigo_br', 'descricao_catmat', 'unidade_fornecimento', 'generico', 'anvisa', 'modalidade_compra', 'tipo_compra', 'capacidade', 'unidade_medida', 'unidade_fornecimento_capacidade', 'cnpj_fornecedor', 'fornecedor', 'cnpj_fabricante', 'fabricante', 'qtd_itens_comprados', 'preco_unitario', 'preco_total']


### Remoção de colunas redundantes e decisão sobre nulos

#### Colunas removidas
| Coluna | Motivo |
|---|---|
| `ano_compra` | Redundante — a informação de ano já está contida em `compra` (datetime) |
| `ano_origem` | Auxiliar de controle usado na concatenação — não agrega ao Data Warehouse |

#### Decisão sobre nulos em `insercao`
Os 2.008 nulos restantes na coluna `insercao` são **mantidos intencionalmente**.

`insercao` representa a data em que o registro foi inserido no sistema BPS.
Quando essa informação não está disponível na fonte original, o valor correto
no banco de dados é `NULL` — não faz sentido imputar uma data fictícia.

Esses registros **não serão removidos** pois todas as demais informações da
compra (produto, instituição, fornecedor, preço, quantidade) estão presentes
e são válidas para análise. Remover 2.008 linhas por ausência de um campo
auxiliar representaria perda desnecessária de 2,38% dos dados.

### Resultado Final 

In [9]:
display(df_clean.info())
display(df_clean.head())

<class 'pandas.DataFrame'>
RangeIndex: 84465 entries, 0 to 84464
Data columns (total 24 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   nome_instituicao                 84465 non-null  str           
 1   esfera                           84465 non-null  str           
 2   cnpj_instituicao                 84465 non-null  str           
 3   municipio_instituicao            84465 non-null  str           
 4   uf                               84465 non-null  str           
 5   compra                           84465 non-null  datetime64[us]
 6   insercao                         82457 non-null  datetime64[us]
 7   codigo_br                        84465 non-null  int64         
 8   descricao_catmat                 84465 non-null  str           
 9   unidade_fornecimento             84465 non-null  str           
 10  generico                         84465 non-null  str           
 11  

None

,nome_instituicao,esfera,cnpj_instituicao,municipio_instituicao,uf,compra,insercao,codigo_br,descricao_catmat,unidade_fornecimento,...,capacidade,unidade_medida,unidade_fornecimento_capacidade,cnpj_fornecedor,fornecedor,cnpj_fabricante,fabricante,qtd_itens_comprados,preco_unitario,preco_total
0,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18478187000107,MARABA,PA,2023-01-01,2024-03-15,423975,"PIPETA, TIPO:PASTEUR, CAPACIDADE:3 ML, MATERIA...",UNIDADE,...,0.0,Não informado,UNIDADE,05323167000107,CIRUBEL COMERCIO E REPRESENTACOES DE PRODUTOS ...,57893521000132,PERFITECNICA PERFIS TECNICOS DE BORRACHA LTDA,36,45.00,1620.0
1,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18478187000107,MARABA,PA,2023-01-01,2024-03-15,457503,"CORANTE, TIPO :PARDO DE BISMARCK, CARACTERÍSTI...",UNIDADE,...,0.0,Não informado,UNIDADE,07944100000115,PROC9 INDUSTRIA QUIMICA LTDA,07944100000115,PROC9 INDUSTRIA QUIMICA LTDA,4,190.00,760.0
2,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18478187000107,MARABA,PA,2023-01-01,2024-03-15,382447,"REAGENTE PARA DIAGNÓSTICO CLÍNICO 5, TIPO:ALBU...",FRASCO,...,10.0,ML,FRASCO 10.00 ML,05048534000101,NORTEMED DISTRIBUIDORA DE PRODUTOS MEDICOS LTDA,50657402000131,EBRAM PRODUTOS LABORATORIAIS LTDA,20,33.38,667.6
3,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18478187000107,MARABA,PA,2023-01-01,2024-03-15,470305,"MEIO DE CULTURA - CÉLULA E TECIDO, TIPO:MEIO H...",GRAMA,...,0.0,Não informado,GRAMA,05323167000107,CIRUBEL COMERCIO E REPRESENTACOES DE PRODUTOS ...,73636391000109,NEWPROV PRODUTOS PARA LABORATORIO LTDA,80,25.00,2000.0
4,FUNDO MUNICIPAL DE SAUDE DE MARABA,MUNICIPAL,18478187000107,MARABA,PA,2023-01-01,2024-03-15,479641,"ADAPTADOR USO MÉDICO, APLICAÇÃO:P/ AGULHA DE C...",UNIDADE,...,0.0,Não informado,UNIDADE,05048534000101,NORTEMED DISTRIBUIDORA DE PRODUTOS MEDICOS LTDA,48740849000128,CRAL ARTIGOS PARA LABORATORIO LTDA,50,38.71,1935.5


## 4. Modelagem dimensional e carga no PostgreSQL

Esta etapa transforma o `df_clean` em um **Esquema Estrela** e carrega
as tabelas resultantes no banco de dados PostgreSQL local.

### Dimensões do nosso modelo
- **`dim_produto`** — *O que foi comprado?* (CATMAT, descrição, unidade, genérico, Anvisa, capacidade)
- **`dim_instituicao`** — *Quem comprou?* (nome, CNPJ, município, UF, esfera)
- **`dim_fornecedor`** — *Quem vendeu?* (CNPJ e nome do fornecedor e fabricante)
- **`dim_tempo`** — *Quando foi comprado?* (data, dia, mês, ano, trimestre, dia da semana)

### Tabela fato
- **`fato_compras`** — registros de compras com as chaves estrangeiras das
  dimensões e as métricas (quantidade, preço unitário, preço total)

In [10]:
# SEÇÃO 4 — MODELAGEM DIMENSIONAL E CARGA


# ── 4.1 Criação das dimensões ────────────────────────────────

# 1. Dimensão Produto (O que foi comprado?)
colunas_produto = [
    'codigo_br', 'descricao_catmat', 'unidade_fornecimento',
    'generico', 'anvisa', 'capacidade', 'unidade_medida',
    'unidade_fornecimento_capacidade'
]

dim_produto = df_clean[colunas_produto].drop_duplicates().reset_index(drop=True)
dim_produto = dim_produto.reset_index().rename(columns={'index': 'id_produto_sk'})

print(f"Dimensão Produto: {len(dim_produto)} produtos únicos")

# 2. Dimensão Instituição (Quem comprou?)
colunas_instituicao = [
    'cnpj_instituicao', 'nome_instituicao',
    'municipio_instituicao', 'uf', 'esfera'
]

dim_instituicao = df_clean[colunas_instituicao].drop_duplicates().reset_index(drop=True)
dim_instituicao = dim_instituicao.reset_index().rename(columns={'index': 'id_instituicao_sk'})

print(f"Dimensão Instituição: {len(dim_instituicao)} instituições únicas")

# 3. Dimensão Fornecedor (Quem vendeu?)
colunas_fornecedor = [
    'cnpj_fornecedor', 'fornecedor',
    'cnpj_fabricante', 'fabricante'
]

dim_fornecedor = df_clean[colunas_fornecedor].drop_duplicates().reset_index(drop=True)
dim_fornecedor = dim_fornecedor.reset_index().rename(columns={'index': 'id_fornecedor_sk'})

print(f"Dimensão Fornecedor: {len(dim_fornecedor)} fornecedores únicos")

# 4. Dimensão Tempo (Quando foi comprado?)
datas_unicas = df_clean[['compra']].drop_duplicates().sort_values('compra').reset_index(drop=True)

datas_unicas['dia']        = datas_unicas['compra'].dt.day
datas_unicas['mes']        = datas_unicas['compra'].dt.month
datas_unicas['ano']        = datas_unicas['compra'].dt.year
datas_unicas['trimestre']  = datas_unicas['compra'].dt.quarter
datas_unicas['dia_semana'] = datas_unicas['compra'].dt.day_name(locale='pt_BR.UTF-8')

dim_tempo = datas_unicas.reset_index().rename(columns={'index': 'id_tempo_sk'})

print(f"Dimensão Tempo: {len(dim_tempo)} datas únicas")

Dimensão Produto: 17391 produtos únicos
Dimensão Instituição: 438 instituições únicas
Dimensão Fornecedor: 19604 fornecedores únicos
Dimensão Tempo: 892 datas únicas


### Criação da tabela fato

A tabela fato `fato_compras` é construída fazendo `merge` do `df_clean` com
cada uma das dimensões, recuperando as chaves surrogate. Depois selecionamos
apenas as colunas necessárias: as chaves das dimensões e as métricas.

In [11]:
# ── 4.2 Criação da tabela fato ───────────────────────────────

# Merge do df_clean com cada dimensão para recuperar as chaves surrogate
fato = df_clean.merge(dim_produto,     on=colunas_produto,     how='left')
fato = fato.merge(dim_instituicao,     on=colunas_instituicao, how='left')
fato = fato.merge(dim_fornecedor,      on=colunas_fornecedor,  how='left')
fato = fato.merge(dim_tempo[['compra', 'id_tempo_sk']], on='compra', how='left')

# Selecionando apenas as chaves e as métricas
colunas_fato = [
    'id_produto_sk',
    'id_instituicao_sk',
    'id_fornecedor_sk',
    'id_tempo_sk',
    'insercao',
    'modalidade_compra',
    'tipo_compra',
    'qtd_itens_comprados',
    'preco_unitario',
    'preco_total'
]

fato_compras = fato[colunas_fato]
print(f"Fato pronta: {len(fato_compras):,} registros")

Fato pronta: 84,465 registros


### Carga no PostgreSQL

Com o Esquema Estrela construído em memória, carregamos cada tabela no
banco PostgreSQL local utilizando `to_sql()` do pandas. As credenciais
são lidas do arquivo `.env` para não expor dados sensíveis no repositório.

A ordem de carga é importante: primeiro as **dimensões**, depois a **fato**
— pois a fato referencia as chaves das dimensões.

Utilizamos `if_exists='replace'` para que execuções repetidas do notebook
sobrescrevam as tabelas, garantindo um pipeline idempotente durante o
desenvolvimento.

In [12]:
# ── 4.3 Carga no PostgreSQL ──────────────────────────────────

load_dotenv()

DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST     = os.getenv('DB_HOST')
DB_PORT     = os.getenv('DB_PORT')
DB_NAME     = os.getenv('DB_NAME')

DATABASE_URL = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_NAME}?client_encoding=utf8'
)

print("Tentando conectar ao PostgreSQL...")

try:
    engine = create_engine(DATABASE_URL)
    print(f"✅ Conexão estabelecida com {DB_NAME}!\n")
    print("Iniciando carga do Esquema Estrela...\n")

    print("→ Carregando dim_produto...")
    dim_produto.to_sql('dim_produto', con=engine, if_exists='replace', index=False)

    print("→ Carregando dim_instituicao...")
    dim_instituicao.to_sql('dim_instituicao', con=engine, if_exists='replace', index=False)

    print("→ Carregando dim_fornecedor...")
    dim_fornecedor.to_sql('dim_fornecedor', con=engine, if_exists='replace', index=False)

    print("→ Carregando dim_tempo...")
    dim_tempo.to_sql('dim_tempo', con=engine, if_exists='replace', index=False)

    print("→ Carregando fato_compras...")
    fato_compras.to_sql('fato_compras', con=engine, if_exists='replace', index=False)

    print("\n✅ CARGA CONCLUÍDA! As 5 tabelas do Esquema Estrela foram criadas no PostgreSQL.")

except Exception as e:
    print(f"\n❌ ERRO ao carregar os dados:")
    print(f"   {e}")

Tentando conectar ao PostgreSQL...
✅ Conexão estabelecida com bps_dw!

Iniciando carga do Esquema Estrela...

→ Carregando dim_produto...
→ Carregando dim_instituicao...
→ Carregando dim_fornecedor...
→ Carregando dim_tempo...
→ Carregando fato_compras...

✅ CARGA CONCLUÍDA! As 5 tabelas do Esquema Estrela foram criadas no PostgreSQL.


In [13]:
# ── 4.4 Validação da carga ───────────────────────────────────

tabelas = ['dim_produto', 'dim_instituicao', 'dim_fornecedor', 'dim_tempo', 'fato_compras']

print("Contagem de registros por tabela no PostgreSQL:\n")

with engine.connect() as conn:
    for tabela in tabelas:
        resultado = conn.execute(text(f'SELECT COUNT(*) FROM {tabela}'))
        total = resultado.scalar()
        print(f"   {tabela:20s} → {total:,} registros")

print("\n✅ Pipeline ETL concluído com sucesso!")

Contagem de registros por tabela no PostgreSQL:

   dim_produto          → 17,391 registros
   dim_instituicao      → 438 registros
   dim_fornecedor       → 19,604 registros
   dim_tempo            → 892 registros
   fato_compras         → 84,465 registros

✅ Pipeline ETL concluído com sucesso!
